# Milestone 6 Lab, notebook 5 of 8: A judge with a reference answer

## Where we are

| Notebook | What you do |
|---|---|
| 1. Late and corrected files, by hand | Work out by hand what three late files should do to the figures |
| 2. Corrections and incremental runs | Load the late files, recalculate only what they touch, and prove it equals a full rebuild |
| 3. The review desk | Decide the findings that need review, and watch the next run obey |
| 4. Run health and restatements | Read a run's health, and ask the analyst why a reported figure changed |
| **5. A judge with a reference answer** (you are here) | Test a judge on explanations whose grade you know, then use it |
| 6. The end-to-end evaluation | Score the whole system from files to answers, on two question sets |
| 7. The improver | An agent proposes a change to the analyst, you approve, and code keeps it or rolls it back |
| 8. The whole system | Run everything from files to answer, and look back over six milestones |

Model requests in this notebook: **about 12**. A run of this lab with its default small sets costs about 10 US cents.

## What this notebook is for

Code can check that a figure is right. It cannot check that an **explanation** is right. "It fell by $5.00 because a shipments file was corrected" and "it fell by $5.00 because of seasonal demand" have the same figures.

Whether a reason is the right reason is a judgement. So we ask a second AI model, a **judge**.

A judge is only useful if it can be trusted. This notebook gives it a **reference answer** to compare against, and tests it on explanations whose grade you already know, before using it on the analyst.

> This notebook needs `OPENROUTER_API_KEY` in your `.env` file. Run notebook 1 first.

In [ ]:
import pandas as pd
from lab_helpers import LAB_E2E, show_doc

from retail_profit_stack.e2e import build_everything, open_stores

workspace = build_everything(LAB_E2E)
stores = open_stores(workspace)

## Step 1: The design

In [ ]:
show_doc("HLD.md", "9. A judge with a reference answer")

## Step 2: The reference answer

A judge asked "is this explanation good?" with nothing to compare against has to rely on its own opinion. Opinions vary. A judge asked "does this explanation give the same reason as this one?" has a much narrower job.

We have the true reason, because code worked it out: it is the run comparison. `reference_reason` writes it as a sentence. No model is involved.

In [ ]:
from retail_profit_stack import tools
from retail_profit_stack.judge import reference_reason

QUESTION = "Why did the fleece pullover's September 2026 contribution profit change since the last run?"

comparison = tools.compare_runs("2026-09", "fleece pullover", store=stores["late"])
reference = reference_reason(comparison)

print(reference)

It is not elegant, and it does not need to be. It is true, and it names the file.

## Step 3: The rubric

A **rubric** tells the judge what each grade means. Ours has four grades.

In [ ]:
from retail_profit_stack.judge import GRADES, JUDGE_MODEL, RUBRIC

print(RUBRIC)
print()
print("Grades as numbers:", GRADES)
print("The judge is:", JUDGE_MODEL)

The judge is a different model from the analyst. A model grading its own work tends to agree with itself. This one was chosen by a short live trial: it and one other low-cost model both gave the expected grade on every test below, and this one was cheaper and faster.

## Step 4: Test the judge before trusting it

Here are six explanations for the same question. We know what grade each deserves.

In [ ]:
tests = {
    "The reference itself": ("A", reference),
    "The same reason, in other words": ("A", "It fell by $76.25. A refund file, refunds_2026-11-09.csv, arrived after the month was closed. Its one refund is dated in September, so it now counts against September."),
    "The main reason, blurred": ("B", "It fell by $76.25 because some more data was loaded since the last run."),
    "Vague": ("C", "The figure changed because the data changed between the two runs."),
    "A different reason": ("F", "It fell by $76.25 because duplicated order lines were set aside by the duplicate check."),
    "It contradicts": ("F", "The figure did not change between the two runs."),
}

`grade_explanation` sends the question, the reference, and one explanation to the judge, and asks for a grade and one sentence of why.

In [ ]:
from retail_profit_stack.judge import grade_explanation, make_judge

judge = make_judge()

rows = []
total_cost = 0.0
for name, (expected, explanation) in tests.items():
    verdict, cost = grade_explanation(judge, QUESTION, reference, explanation)
    total_cost += cost
    rows.append({"explanation": name, "we expect": expected, "the judge says": verdict.grade, "why": verdict.why})

pd.set_option("display.max_colwidth", 110)
print("Six grades cost", round(total_cost, 5), "US dollars")
pd.DataFrame(rows)

Compare the two grade columns. A blurred explanation may get A, B, or C from run to run: where "mostly right" ends is a matter of opinion, even for a person. In the first trial the judge gave the blurred one an A, so the rubric was tightened to say an A needs the specific cause. The important rows are the ends. The true reason gets A, and a wrong reason gets F.

Look at the fifth row again. "Duplicated order lines were set aside" is a **true statement about this business**: it is the real reason for a different change. It reads well. Without a reference, a judge might have liked it. With one, it is plainly the wrong reason.

If a grade came out differently, run the cell again, and read the judge's `why`.

## Step 5: Judge the analyst

Now a real answer. Ask the analyst the question, and give its explanation to the judge.

In [ ]:
from lab_helpers import show_answer

from retail_profit_stack.agent import run_analyst

run = run_analyst(QUESTION, store=stores["late"])

show_answer(run.answer)

In [ ]:
explanation = f"{run.answer.headline} {run.answer.narrative}"

verdict, cost = grade_explanation(judge, QUESTION, reference, explanation)

print("Grade:", verdict.grade, f"({GRADES[verdict.grade]} out of 3)")
print("Why:  ", verdict.why)
print("Verifier:", run.verification.passed, [check.name for check in run.verification.checks if not check.passed])

If the verifier lists a failed check, run the two cells again: the analyst is good, not perfect.

The answer now has two kinds of check. The **verifier**, in code, checked every figure, the run, and the warnings. The **judge**, a model, checked the reason. Neither could do the other's job.

## Step 6: What the judge does not do

The judge grades only "why did it change" questions, because only they have a reference. It does not check figures: code does that exactly and for free.

And the judge can be wrong. So its grade is one score among several, and it never decides anything by itself.

## What we now know

- Code checks figures. Whether a **reason** is right is a judgement, so a model makes it.
- A judge with a **reference answer** has a narrow job: does this explanation give the same reason?
- Our reference is written by code from the run comparison, so it is true.
- **Test a judge on known cases before trusting it.** Ours gave the expected grade on all six.
- The judge is a different model from the analyst, and its grade is one score among several.

**Next:** notebook 6. With the verifier and the judge, we can score the whole system, from files to answers.